# Tugas 2 - Text Preprocessing, TF-IDF, ICSDF, dan PCA

Pada tugas ini dilakukan pengolahan terhadap 200 berita dari Detik.com yang terdiri dari 100 berita kategori Sport dan 100 berita kategori Finance.

Tahapan yang dilakukan meliputi eksplorasi data, pemeriksaan kualitas teks, normalisasi karakter, tokenisasi, normalisasi kata tidak baku dan istilah asing, pembentukan representasi TF-IDF, pembobotan ICSDF, seleksi fitur, serta reduksi dimensi menggunakan PCA.

Selain itu dilakukan analisis kata unik berdasarkan kategori, analisis kata dengan bobot TF-IDF tertinggi, visualisasi PCA dua dimensi, dan analisis PCA Feature Loading.

## 2. Import Library

Setelah library tersedia, library yang diperlukan diimpor ke dalam notebook.

`pandas` digunakan untuk membaca dan mengolah dataset. `numpy` digunakan untuk perhitungan numerik. `re` digunakan untuk preprocessing teks dengan regular expression. Library dari `scikit-learn` digunakan untuk pembagian data, TF-IDF, dan PCA.

In [3]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

In [ ]:
## 3. Membaca Dataset

Dataset yang digunakan merupakan kumpulan 200 berita yang terdiri dari dua kategori, yaitu Sport dan Finance.

Dataset memiliki tiga kolom utama, yaitu:

- `id` sebagai identitas berita
- `isi_berita` sebagai isi teks berita
- `label` sebagai kategori berita

Masing-masing kategori terdiri dari 100 berita.

In [4]:
df = pd.read_excel("dataset_detik_200_berita.xlsx")

print("Jumlah data :", len(df))
print("\nNama kolom :")
print(df.columns)

print("\nJumlah data per label :")
print(df["label"].value_counts())

Jumlah data : 200

Nama kolom :
Index(['id', 'isi_berita', 'label'], dtype='str')

Jumlah data per label :
label
sport      100
finance    100
Name: count, dtype: int64


In [5]:
## 4. Mengubah Label Menjadi Numerik

Label teks perlu diubah menjadi bentuk numerik agar dapat digunakan pada proses pengolahan data selanjutnya.

Pada tugas ini digunakan aturan:

- `sport` menjadi `1`
- `finance` menjadi `0`

Dengan demikian setiap berita mempunyai label numerik yang mewakili kategorinya.

SyntaxError: invalid syntax (1933235811.py, line 3)

In [6]:
df["label_num"] = df["label"].map({
    "sport": 1,
    "finance": 0
})

df[["id", "label", "label_num"]].head()

,id,label,label_num
0,1,sport,1
1,2,sport,1
2,3,sport,1
3,4,sport,1
4,5,sport,1


In [ ]:
## 5. Menghitung Jumlah Kata Semua Berita

Sebelum dilakukan preprocessing, jumlah kata pada setiap berita dihitung terlebih dahulu.

Perhitungan dilakukan dengan memisahkan teks berdasarkan spasi menggunakan fungsi `split()`.

Hasil perhitungan disimpan pada kolom `jumlah_kata_asli`.

In [7]:
df["jumlah_kata_asli"] = (
    df["isi_berita"]
    .astype(str)
    .apply(lambda x: len(x.split()))
)

print("Total seluruh kata :", df["jumlah_kata_asli"].sum())

df[["id", "label", "jumlah_kata_asli"]].head()

Total seluruh kata : 67076


,id,label,jumlah_kata_asli
0,1,sport,320
1,2,sport,323
2,3,sport,256
3,4,sport,238
4,5,sport,402


In [ ]:
## 6. Kamus Kata Tidak Baku

Kamus kata tidak baku digunakan untuk mengubah kata singkatan atau kata tidak baku menjadi bentuk yang lebih standar.

Contohnya adalah:

- `yg` menjadi `yang`
- `dgn` menjadi `dengan`
- `utk` menjadi `untuk`
- `krn` menjadi `karena`
- `tdk` menjadi `tidak`
- `dr` menjadi `dari`

Normalisasi ini dilakukan agar kata yang mempunyai makna sama tidak dianggap sebagai kata yang berbeda.

In [8]:
kamus_tidak_baku = {
    "gak": "tidak",
    "nggak": "tidak",
    "ga": "tidak",
    "enggak": "tidak",
    "yg": "yang",
    "dgn": "dengan",
    "utk": "untuk",
    "krn": "karena",
    "kalo": "kalau",
    "kalok": "kalau",
    "aja": "saja",
    "udah": "sudah",
    "sdh": "sudah",
    "blm": "belum",
    "tdk": "tidak",
    "dr": "dari",
    "dlm": "dalam",
    "jd": "jadi",
    "bgt": "banget",
    "tp": "tetapi",
    "tapi": "tetapi",
    "karna": "karena",
    "trus": "terus",
    "kmrn": "kemarin",
    "dpt": "dapat",
    "hrs": "harus",
    "sm": "sama",
    "sy": "saya"
}

In [ ]:
## 7. Kamus Bahasa Asing

Kamus bahasa asing digunakan untuk menormalisasi beberapa istilah asing yang terdapat pada berita menjadi istilah bahasa Indonesia.

Kamus disesuaikan dengan dua kategori dataset, yaitu Sport dan Finance.

Contohnya pada kategori Sport:

- `rider` menjadi `pembalap`
- `race` menjadi `balapan`
- `team` menjadi `tim`
- `player` menjadi `pemain`

Sedangkan pada kategori Finance:

- `finance` menjadi `keuangan`
- `market` menjadi `pasar`
- `stock` menjadi `saham`
- `investment` menjadi `investasi`

In [ ]:
kamus_asing = {
    # SPORT
    "rider": "pembalap",
    "race": "balapan",
    "racing": "balap",
    "team": "tim",
    "coach": "pelatih",
    "player": "pemain",
    "match": "pertandingan",
    "winner": "pemenang",
    "season": "musim",
    "training": "latihan",
    "game": "pertandingan",
    "games": "pertandingan",
    "manager": "manajer",
    "champion": "juara",
    "championship": "kejuaraan",
    "league": "liga",
    "score": "skor",
    "goal": "gol",
    "final": "final",

    # FINANCE
    "finance": "keuangan",
    "financial": "keuangan",
    "market": "pasar",
    "stock": "saham",
    "stocks": "saham",
    "sale": "penjualan",
    "price": "harga",
    "business": "bisnis",
    "company": "perusahaan",
    "investment": "investasi",
    "investor": "investor",
    "banking": "perbankan",
    "bank": "bank",
    "economy": "ekonomi",
    "economic": "ekonomi",
    "growth": "pertumbuhan",
    "profit": "keuntungan",
    "loss": "kerugian",
    "revenue": "pendapatan"
}

In [ ]:
## 8. Fungsi Preprocessing

Preprocessing dilakukan untuk membersihkan teks sebelum digunakan dalam proses pembobotan.

Tahapan preprocessing yang dilakukan adalah:

1. Mengubah teks menjadi string.
2. Case folding dengan mengubah seluruh huruf menjadi huruf kecil.
3. Menghapus URL.
4. Menghapus alamat email.
5. Menghapus angka.
6. Menghapus tanda baca, simbol, dan emotikon.
7. Menghapus spasi berlebih.
8. Melakukan tokenisasi menggunakan `split()`.
9. Melakukan normalisasi kata tidak baku.
10. Melakukan normalisasi istilah bahasa asing.
11. Menggabungkan kembali token menjadi teks.

Tahapan tersebut bertujuan menghasilkan teks yang lebih bersih dan konsisten.

In [9]:
def preprocessing(text):
    text = str(text)

    # Case folding
    text = text.lower()

    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Hapus email
    text = re.sub(r'\S+@\S+', ' ', text)

    # Hapus angka
    text = re.sub(r'\d+', ' ', text)

    # Hapus karakter selain huruf
    text = re.sub(r'[^a-zA-ZÀ-ÿ\s]', ' ', text)

    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenisasi
    tokens = text.split()

    # Normalisasi kata
    hasil = []

    for token in tokens:
        if token in kamus_tidak_baku:
            token = kamus_tidak_baku[token]

        if token in kamus_asing:
            token = kamus_asing[token]

        hasil.append(token)

    # Gabungkan kembali
    return " ".join(hasil)

In [10]:
## 9. Menerapkan Preprocessing ke Semua Berita

Setelah fungsi preprocessing dibuat, fungsi tersebut diterapkan pada seluruh isi berita.

Hasil preprocessing disimpan pada kolom baru bernama `berita_clean`.

In [15]:
df["berita_clean"] = df["isi_berita"].apply(preprocessing)

df[["isi_berita", "berita_clean"]].head()

NameError: name 'kamus_asing' is not defined

In [14]:
df["jumlah_kata_clean"] = (
    df["berita_clean"]
    .apply(lambda x: len(x.split()))
)

print("Kolom jumlah_kata_clean berhasil dibuat.")

KeyError: 'berita_clean'

KeyError: 'berita_clean'